<a href="https://colab.research.google.com/github/flzhfi/deep-learning-assignments/blob/main/11%EC%A3%BC%EC%B0%A8_2353764_%EC%8B%A0%EC%98%88%EC%A7%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**[11주차]실습**
- 교재를 참고하여 아래의 실습을 수행하시오.
- 모든 코드의 결과를 출력하여, .ipynb의 링크를 **[11주차]/[11주차]과제**에 제출하시오.\
(실습 제출 예시: 11주차_2020XXXX_이름.ipynb 코드 링크)

In [ ]:
print("2353764, 신예진")

2353764, 신예진


In [ ]:
# 설치 후 세션 재시작, 재시작 후에는 해당 셀 스킵
!pip install numpy==1.24.1 regex==2017.4.5 requests==2.27.1 tqdm==4.64.0 fire==0.5.0

  Using cached numpy-1.24.1.tar.gz (10.9 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
from google.colab import auth

auth.authenticate_user()
!gcloud config get-value account

snayou2@gmail.com


In [ ]:
# google drive 연결
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys

# Add the directory containing utils.py to sys.path
sys.path.append('/content/drive/MyDrive/deeplearning/week11/')


In [ ]:
!pip install --upgrade regex

###**예제 1) picoGPT의 decoder를 작성해본다.**
- **[실습목표]** picoGPT의 decode를 작성하며, transformer의 구조를 파악해본다.


In [ ]:
import numpy as np
from tqdm import tqdm
from utils import load_encoder_hparams_and_params


#### **1) main() 메소드**
- GPT-2모델을 사용하여 텍스트 생성을 실행할 때 필요한 입력값을 설정
- 아래의 각 매개변수의 역할을 서술하시오.
  - prompt(str): 시작 문장
  - n_tokens_to_generate(int):몇 개의 토큰을 추가로 생성할지 결정함

In [ ]:
import numpy as np # numpy가 필요하므로 추가

def main(prompt: str, n_tokens_to_generate: int = 40, model_size: str = "124M", models_dir: str = "models", temperature: float = 1.0):
    encoder, hparams, params = load_encoder_hparams_and_params(model_size, models_dir)
    input_ids = encoder.encode(prompt)  # 입력 문장을 숫자로 변환
    assert len(input_ids) + n_tokens_to_generate < hparams["n_ctx"]
    output_ids = generate(inputs=np.array([input_ids]), params=params,
                          n_head=hparams["n_head"], n_tokens_to_generate=n_tokens_to_generate,
                          temperature=temperature)  # GPT-2 단어 생성
    output_text = encoder.decode(output_ids[0])  # 생성된 숫자를 문자로 변환
    return output_text

#### **2) generate() 메소드**
- 주어진 입력 시퀀스에 대해 추가적인 토큰을 생성하는 함수
- generate()함수 내에서 gpt2 함수를 통해 입력 시퀀스에 대한 예측을 수행
- 아래의 각 변수의 역할에 대해 서술
  - logits: 각 토큰이 나올 가능성을 나타내는 값


In [ ]:
def generate(inputs, params, n_head, n_tokens_to_generate, temperature=1.0):
    for _ in tqdm(range(n_tokens_to_generate), "generating"):
        logits = gpt2(inputs, params, n_head) # 입력 문장을 보고 다음 토큰의 후보에 대한 점수를 계산
        logits = logits[:, -1, :] / temperature
        next_id_probs = softmax(logits)
        next_id = np.array([np.random.choice(len(next_id_probs[0]), p=next_id_probs[0])])

        inputs = np.concatenate([inputs, next_id[:, np.newaxis]], axis=1)  # 선택된 토큰을 기존 문장 뒤에 붙임


    return inputs # 최종 생성 문장 반환


#### **3) gpt2() 메소드**
- 입력 시퀀스에 대해 예측을 수행

In [ ]:
def gpt2(inputs, params, n_head):
    wte = params['wte']
    wpe = params['wpe']
    blocks_params = params['blocks'] # 각 블록의 파라미터 리스트
    ln_f_params = params['ln_f']

    # 입력 토큰 -> 임베딩
    x = wte[inputs] + wpe[:inputs.shape[1]]

    # Transfomer block 반복
    for block_params in blocks_params:

        norm_x_1 = layer_norm(x, **block_params['ln_1'])
        attn_output = mha(norm_x_1, **block_params['attn'], n_head=n_head)
        x = x + attn_output

        norm_x_2 = layer_norm(x, **block_params['ln_2'])
        mlp_output = ffn(norm_x_2, **block_params['mlp'])
        x = x + mlp_output

    x = layer_norm(x, **ln_f_params)
    logists = x @ wte.T

    # 최종 Layer Normalization
    return logists

#### **4) transformer_block()메소드**
- 멀티헤드 어텐션(Multi Head Attention, MHA)와 피드-포워드 신경망(Feed Foward Network)를 포함하는 트렌스포머 블록을 구현한 함수



In [ ]:
def transformer_block(x, mlp, attn, ln_1, ln_2, n_head):
    x = x + attn(ln_1(x), n_head)
    x = x + mlp(ln_2(x))
    return x

#### **5) mha()**
- 멀티헤드 어텐션함수는 입력 벡터를 여러 개의 어텐션 헤드로 나누어 병렬로 어텐션을 계산한 후, 그 결과를 결합하는 역할
- 입력 벡터를 여러개의 어텐션 헤드로 나누는 이유는 무엇인가?
- : 문장 안의 다양한 관계를 동시에 학습하기 위해서


In [ ]:
def mha(x, c_attn, c_proj, n_head):
    x = linear(x, **c_attn)

    qkv_heads = list(map(lambda x: np.split(x, n_head, axis=-1), np.split(x, 3, axis=-1)))
    causal_mask = (1 - np.tri(x.shape[1], dtype=x.dtype)) * -1e10

    out_heads = [attention(q, k, v, causal_mask) for q, k, v in zip(*qkv_heads)]
    x = linear(np.concatenate(out_heads, axis=-1), **c_proj)
    return x

#### **6) attention() 메소드**
- 어텐션 함수는 주어진 쿼리(Q), 키(K), 값(V) 및 마스크를 사용하여 어텐션 가중치를 계산하는 함수이다.


In [ ]:
def attention(q, k, v, mask):
    return softmax(q @ k.transpose(0, 2, 1) / np.sqrt(q.shape[-1]) + mask) @ v

#### **7) ffn() 메소드**
- 피드-포워드 신경망 함수는 트렌스포머 모델에서 사용되는 구성 요소 중 하나로, 입력 벡터에 대해 두 번의 선형 변환과 활성화 함수를 적용하여 출력 벡터를 생성하는 함수이다.
- 멀티헤드 어텐션 이후에 적용.

In [ ]:
def ffn(x, c_fc, c_proj):
    return linear(gelu(linear(x, **c_fc)), **c_proj)

#### **8, 9, 10) linear(), layer_norm(), gelu(), softmax() 메소드**
- 각각 선형 변환 함수, 레이어 정규화 함수, GELU 활성화 함수, Softmax 함수를 의미하며, 각 함수의 역할은 아래의 설명과 같다.
- **linear()**: 입력 벡터에 대해 선형 변환을 수행.
- **layer_norm()**: 각 레이어의 출력에 정규화를 수행하여 학습 안정성을 높이고, 수렴 속도를 향상.
- **gelu()**: 입력 값을 비선형적으로 변환하여 모델의 표현력을 향상
- **softmax()**: 입력 벡터를 확률 분포로 변환하는 함수, 출력 벡터의 요소들을 0과 1 사이의 값으로 매핑 시키며, 전체 합이 1이 되도록 한다.

In [ ]:
def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def layer_norm(x, g, b, eps: float = 1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    variance = np.var(x, axis=-1, keepdims=True)
    return g * (x - mean) / np.sqrt(variance + eps) + b

def linear(x, w, b):
    return x @ w + b

#### **11) prompt 입력 후 결과 출력**


In [ ]:
if __name__ == "__main__":
    print(main(prompt="Once upon a time", temperature=0.7))
    # prompt를 여러가지로 변경하며, 테스트 해보기(선택사항)

generating: 100%|██████████| 40/40 [01:37<00:00,  2.45s/it]

Once upon a time, these men were in a good place, and were in the best shape, but that time came and came and came. And as I was going away, they were there, and were in a
